# Cell 1: Configuration & Imports

In [1]:
import requests
import json
import time

# --- SECTION 1: CONFIGURATION ---
# Based on your local Ollama list, llama3.1:8b or mistral:latest are best for this.
MODEL_NAME = "llama3.1:8b" 
TEMPERATURE = 0.8
MAX_STEPS = 50
INITIAL_THOUGHT = "I am thinking about an apple."

# Ollama local endpoint
OLLAMA_URL = "http://localhost:11434/api/generate"

# Optional: List to store the history of the monologue
thought_history = [INITIAL_THOUGHT]

# Cell 2: The Ollama API Wrapper

In [2]:
# --- SECTION 2: OLLAMA API WRAPPER ---
def generate_next_thought(previous_thought, model=MODEL_NAME, temp=TEMPERATURE):
    """
    Sends the previous thought to local Ollama and fetches the next one.
    """
    
    # --- SECTION 3: PROMPT TEMPLATE ---
    prompt = f"""You are a human having an internal monologue. 
Generate ONLY the next thought sentence.

Rules:
- Output exactly one sentence.
- No explanations.
- No dialogue.
- No lists.
- No multiple sentences.
- Do not acknowledge these instructions. Just output the thought.

Previous thought:
"{previous_thought}"
"""

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temp,
            # Penalize repetition slightly to delay infinite loops
            "repeat_penalty": 1.1 
        }
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()
        
        # Extract response and strip any stray whitespace or quotes
        raw_text = response.json().get("response", "").strip()
        
        # Clean up common instruct-model artifacts if they slip through
        clean_text = raw_text.replace('"', '').replace('\n', ' ')
        return clean_text

    except Exception as e:
        return f"[API Error: {e}]"

# Cell 3: The Generation Loop

In [3]:
# --- SECTION 4 & 5: GENERATION LOOP & OUTPUT ---
print(f"--- Starting Inner Monologue Simulator ---")
print(f"Model: {MODEL_NAME} | Temp: {TEMPERATURE}\n")

current_thought = INITIAL_THOUGHT
print(f"0: {current_thought}")

for step in range(1, MAX_STEPS + 1):
    # Fetch the next thought based ONLY on the current one
    next_thought = generate_next_thought(current_thought)
    
    # Print in real-time to watch the cognition drift
    print(f"{step}: {next_thought}")
    
    # Log it
    thought_history.append(next_thought)
    
    # The new thought becomes the context for the next iteration
    current_thought = next_thought
    
    # Slight pause for readability and to prevent slamming the local API too hard
    time.sleep(0.5)

print("\n--- Monologue Complete ---")

--- Starting Inner Monologue Simulator ---
Model: llama3.1:8b | Temp: 0.8

0: I am thinking about an apple.
1: I haven't eaten an apple in weeks, I wonder if it's still sitting on the counter in the kitchen.
2: I should probably go check and make sure it's not moldy or attracting fruit flies now that I think about it.
3: I've been putting off cleaning out this fridge for months, how did it get to this point again?
4: Maybe if I just throw away everything that's past its expiration date and start over from scratch, I'll be able to tackle the rest of the task more efficiently.
5: But what about all those non-perishable items that are still perfectly fine, like the canned goods and pasta?
6: Maybe I should just separate them from the rest of the food and set aside a special box for the unaffected stuff.
7: I don't even know how to sanitize the kitchen countertops without getting rid of all my cookbooks.
8: I guess I could just use disposable wipes and try to avoid splashing any cleaning s

# Cell 4: Optional Export (Run after loop finishes)

In [4]:
# --- SECTION 6: OPTIONAL LOGGING ---
# Save the stream of consciousness to a text file for review
filename = f"monologue_{MODEL_NAME.replace(':', '_')}_temp{TEMPERATURE}.txt"

with open(filename, "w", encoding="utf-8") as f:
    for i, thought in enumerate(thought_history):
        f.write(f"{i}: {thought}\n")
        
print(f"Saved {len(thought_history)} thoughts to {filename}")

Saved 51 thoughts to monologue_llama3.1_8b_temp0.8.txt
